In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numba import cuda, float64, complex128
from numba.cuda import jit as cuda_jit
import math

import few

from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode import KerrEccEqFlux
from few.amplitude.ampinterp2d import AmpInterpKerrEccEq
from few.summation.interpolatedmodesum import InterpolatedModeSum 


from few.utils.ylm import GetYlms

from few import get_file_manager

from few.waveform import FastKerrEccentricEquatorialFlux, GenerateEMRIWaveform

from few.utils.geodesic import get_fundamental_frequencies

from few.utils.constants import YRSID_SI

import os
import sys

# Change to the desired directory
os.chdir('/nfs/home/svu/e1498138/localgit/FEWNEW/work/')

# Add it to Python path
sys.path.insert(0, '/nfs/home/svu/e1498138/localgit/FEWNEW/work/')

import GWfuncs
import loglike
import modeselector
import dynesty
# import gc
# import pickle
import cupy as cp

# tune few configuration
cfg_set = few.get_config_setter(reset=True)
cfg_set.set_log_level("info")

# GPU configuration 
use_gpu = True
force_backend = "cuda12x"  
dt = 10     # Time step
T = 0.25     # Total time



In [ ]:
# keyword arguments for inspiral generator 
inspiral_kwargs={
        "func": 'KerrEccEqFlux',
        "DENSE_STEPPING": 0, #change to 1/True for uniform sampling
        "include_minus_m": True, 
}

# keyword arguments for inspiral generator 
amplitude_kwargs = {
    "force_backend": force_backend,
}

# keyword arguments for Ylm generator (GetYlms)
Ylm_kwargs = {
    "force_backend": force_backend,
}

# keyword arguments for summation generator (InterpolatedModeSum)
sum_kwargs_sep = {
    "force_backend":force_backend,
    "pad_output": True,
    "separate_modes": True
}
sum_kwargs_comb = {
    "force_backend":force_backend,
    "pad_output": True,
}

print("Creating GenerateEMRIWaveform class...")
print("Creating waveform generator for separate modes...")

waveform_gen_sep = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs_sep,
    use_gpu=use_gpu
)

print("Creating waveform generator for combined modes...")
waveform_gen_comb = GenerateEMRIWaveform(
    FastKerrEccentricEquatorialFlux, 
    frame='detector',
    inspiral_kwargs=inspiral_kwargs, 
    amplitude_kwargs=amplitude_kwargs, 
    Ylm_kwargs=Ylm_kwargs,
    sum_kwargs=sum_kwargs_comb,
    use_gpu=use_gpu 
)

In [ ]:
gwf = GWfuncs.GravWaveAnalysis(T, dt)

In [ ]:
#Generating data (true)

m1 = 1e6
m2 = 3e1
a = 0.7
p0 = 7.5
e0 = 0.4
xI0 = 1.0 #NOTE: fixed, equatorial
dist = 3 
qS = 0.5 
phiS = 1
qK = 1 #NOTE: fixed, degenerate
phiK = 1.5 #NOTE: fixed, degenerate
# Phases
Phi_phi0 = 0.4
Phi_theta0 = 0.0 # NOTE: fixed, equatorial
Phi_r0 = 0.5

params_star = (m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0)

In [ ]:
data_wave = waveform_gen_comb(m1, m2, a, p0, e0, xI0, dist, qS, phiS, qK, phiK, Phi_phi0, Phi_theta0, Phi_r0,  T=T, dt=dt)

In [ ]:
plt.plot(data_wave.get().real, label='real')
plt.plot(data_wave.get().imag, label='imag')
plt.legend(loc='upper right')
plt.xlim(0, 100)
plt.show()

# Loglike func

In [ ]:
loglike_obj = loglike.LogLike(params_star, waveform_gen, gwf, M_init=30, verbose=True, distance_threshold=1.5)

In [ ]:
loglike_obj.signal

In [ ]:
def parameter_space_search_example(n_samples=10):
    print(f"Running parameter space search with {n_samples} samples...")

    # 3sigma
    # intrinsic parameters ranges
    m1_range = (9.9999999987e+05, 1.0000000001e+06)
    m2_range = (2.9999921753e+01, 3.0000078247e+01)
    a_range = (6.9989929349e-01, 7.0010070651e-01)
    p0_range = (7.4994459434e+00, 7.5005540566e+00)
    e0_range = (3.9999126130e-01, 4.0000873870e-01) #1sigma
    dist_range = (2.9566301838e+00, 3.0433698162e+00)
    # detector frame angles ranges
    qS_range = (4.7685409100e-01, 5.2314590900e-01)   
    phiS_range = (9.7607316244e-01, 1.0239268376e+00) 
    # phase ranges
    Phi_phi0_range = (3.6664072881e-01, 4.3335927119e-0)    
    Phi_r0_range = (4.8936765646e-01, 5.1063234354e-01)     

    # Seed for reproducibility
    np.random.seed(7)

    for i in range(n_samples):
        # Sample masses log-uniformly, others uniformly
        m1 = 10**(np.random.uniform(np.log10(m1_range[0]),np.log10(m1_range[1])))
        m2 = 10**(np.random.uniform(np.log10(m2_range[0]),np.log10(m2_range[1])))
        params = np.array([
            m1,
            m2,
            np.random.uniform(*a_range),
            np.random.uniform(*p0_range),
            np.random.uniform(*e0_range),
            xI0,  # xI0 fixed, equatorial
            np.random.uniform(*dist_range),
            np.random.uniform(*qS_range),
            np.random.uniform(*phiS_range),
            qK, # qK fixed, degenerate
            phiK, # phiK fixed, degenerate
            np.random.uniform(*Phi_phi0_range),
            Phi_theta0,  # Phi_theta0 fixed, equatorial 
            np.random.uniform(*Phi_r0_range)
        ])

        try:
            # Evaluate likelihood
            print(f" === Parameters for sample {i+1}: {params} ===")
            f_stat = loglike_obj(params)
            print(f"Sample {i+1}/{n_samples}: f_stat = {f_stat}")

        except Exception as e:
            print(f"Error in evaluation {i+1}: {e}")
            continue

In [ ]:
gener